In [16]:
import pandas as pd
import numpy as np

## Data Cleaning Process

In [23]:
base_path = "/Users/hendrikalbrecht/Neuefische/Capstone-Project/data"

In [28]:
def clean_eurostat_airport_file(filepath, sheet_name="Sheet 1"):
    # Einlesen
    df_raw = pd.read_excel(
        filepath,
        sheet_name=sheet_name,
        header=8
    )

    # Kopie für Cleaning
    df_clean = df_raw.copy()

    # AIRP_PR (Labels) entfernen
    df_clean = df_clean.iloc[1:].reset_index(drop=True)

    # Unnamed-Spalten entfernen
    df_clean = df_clean.loc[
        :,
        ~df_clean.columns.str.startswith("Unnamed")
    ]

    # TIME in origin und destination splitten
    df_clean[["origin", "destination"]] = (
        df_clean["TIME"].str.split(" - ", n=1, expand=True)
    )

    # TIME entfernen
    df_clean = df_clean.drop(columns="TIME")

    # origin und destination nach vorne stellen
    df_clean = df_clean[
        ["origin", "destination"] +
        [col for col in df_clean.columns if col not in ["origin", "destination"]]
    ]

    # ":" durch NaN ersetzen
    df_clean = df_clean.replace(":", np.nan)

    # Special-value-Zeile entfernen
    df_clean = df_clean[df_clean["origin"] != "Special value"]

    # not available-Zeile entfernen
    df_clean = df_clean[
        ~df_clean.iloc[:, 2:].eq("not available").any(axis=1)
    ]
    
    # break-in-time-series-Zeile entfernen
    df_clean = df_clean[
        ~df_clean.iloc[:, 2:].eq("break in time series").any(axis=1)
]
    # leere Zeilen entfernen
    df_clean = df_clean.dropna(how="all").reset_index(drop=True)
    
    # Index neu setzen
    df_clean = df_clean.reset_index(drop=True)

    # 8. Datentypen anpassen
    month_columns = df_clean.columns[2:]

    df_clean[month_columns] = (
        df_clean[month_columns]
        .astype("Int64")
    )

    return df_raw, df_clean

In [31]:
passengers_raw = {}
passengers_clean = {}

for year in range(2017, 2025):
    filepath = f"{base_path}/NL_passengers_{year}.xlsx"
    
    passengers_raw[year], passengers_clean[year] = clean_eurostat_airport_file(filepath)
    
    print(year, passengers_clean[year].shape)

/opt/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


2017 (495, 14)
2018 (495, 14)
2019 (495, 14)


/opt/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


2020 (496, 14)
2021 (495, 14)
2022 (495, 14)
2023 (495, 14)
2024 (495, 14)


/opt/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [32]:
passengers_clean[2017].info()

<class 'pandas.DataFrame'>
RangeIndex: 495 entries, 0 to 494
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   origin       495 non-null    str  
 1   destination  495 non-null    str  
 2   2017-01      254 non-null    Int64
 3   2017-02      254 non-null    Int64
 4   2017-03      257 non-null    Int64
 5   2017-04      277 non-null    Int64
 6   2017-05      275 non-null    Int64
 7   2017-06      275 non-null    Int64
 8   2017-07      277 non-null    Int64
 9   2017-08      274 non-null    Int64
 10  2017-09      275 non-null    Int64
 11  2017-10      276 non-null    Int64
 12  2017-11      259 non-null    Int64
 13  2017-12      255 non-null    Int64
dtypes: Int64(12), str(2)
memory usage: 60.1 KB


In [33]:
passengers_clean[2020].tail()

,origin,destination,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12
491,ROTTERDAM airport,LONDON/CITY airport,11709,11683,4048,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1
492,ROTTERDAM airport,LONDON HEATHROW airport,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
493,ROTTERDAM airport,EDINBURGH airport,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
494,ROTTERDAM airport,LONDON STANSTED airport,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
495,Observation flags:,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [34]:
passengers_clean[2024].head()

,origin,destination,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
0,AMSTERDAM/SCHIPHOL airport,ABU DHABI INTERNATIONAL airport,17425,17018,17754,17511,17889,17733,17771,19044,18301,17664,18633,18034
1,AMSTERDAM/SCHIPHOL airport,DUBAI INTERNATIONAL airport,84484,85688,84948,75681,77175,72086,83419,82348,76943,87356,88525,92434
2,AMSTERDAM/SCHIPHOL airport,BONAIRE/FLAMINGO airport,18414,17994,18730,16069,15110,12860,14878,15394,12166,15171,16599,16551
3,AMSTERDAM/SCHIPHOL airport,CURACAO/AEROPUERTO HATO airport,65783,61620,62815,60458,57792,49548,67733,69039,53751,54678,56559,60851
4,AMSTERDAM/SCHIPHOL airport,ST. MAARTEN/PRINCESS JULIANA airport,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [35]:
passengers_clean[2023].describe()

,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12
count,253.0,256.0,256.0,268.0,268.0,263.0,271.0,271.0,272.0,272.0,251.0,252.0
mean,16413.450593,15781.101562,19369.640625,20944.160448,22739.380597,23163.479087,23639.571956,23825.236162,22921.819853,23282.753676,20138.12749,20298.214286
std,17211.887032,16918.153198,20319.579113,21342.734183,22426.345488,22544.785171,22827.577831,22886.565987,22394.967829,23473.728903,21408.184824,21201.780358
min,0.0,0.0,16.0,253.0,1.0,2.0,4.0,10.0,14.0,16.0,10.0,4.0
25%,5474.0,5101.75,6574.5,7135.5,8175.5,8329.0,8122.5,8697.5,8284.75,7870.0,6973.0,7104.0
50%,10375.0,10101.0,12607.0,13397.5,15016.5,14975.0,15425.0,15650.0,15422.0,15325.0,13466.0,12402.0
75%,21207.0,19671.5,24151.5,25732.5,27499.5,28703.5,29697.0,30752.5,29144.25,27970.5,24094.5,24892.75
max,98940.0,93174.0,111262.0,122667.0,129368.0,131826.0,136194.0,140025.0,134701.0,140549.0,124279.0,121604.0


## Important (January 2023)
Passenger volumes per route are highly skewed. While the median route transported around 10,000 passengers in January 2023, a small number of very busy routes increased the average to more than 16,000 passengers. This indicates that traffic is concentrated on relatively few major connections.